# Notebook BusOps

### Generierung von fehlerhaften Rohdaten zur Bereinigung

In [ ]:
import os
import random 
import pandas as pd
from datetime import datetime, timedelta

random.seed(7)

rawdata = "/lakehouse/default/Files/raw"
os.makedirs(rawdata, exist_ok=True)

### Fahrzeuge und Werkstätte erzeugen

In [ ]:
import random
import pandas as pd

depots = ["Nord", "South", "East", "West"]
bustype = [("Solobus", 12), ("Solobus", 12), ("Solobus", 18), ("Midibus", 9)]

fahrzeuge = []
for i in range(1, 181):
    
    typ, laenge = random.choice(bustype)
    
    baujahr = random.randint(2009, 2024)
    
    if baujahr >= 2021: 
        antrieb = random.choices(["Diesel", "Elektro", "Hybrid"], weights=[0.3, 0.5, 0.2])[0]
    else: 
        antrieb = random.choices(["Diesel", "Hybrid"], weights=[0.85, 0.15])[0]
        
    fahrzeuge.append({
        "fahrzeug_id": "BUS%03d" % i,
        "typ": typ,
        "laenge_m": laenge,
        "baujahr": baujahr,
        "antrieb": antrieb,
        "depot": random.choice(depots),
        "start_km_stand": random.randint(20000, 780000)
    })

df_fahrz = pd.DataFrame(fahrzeuge)

werkstaette = [
    {"werkstatt_id": "W1", "standort": "Hof Nord", "hebebuehne": 6, "eigenbetrieb": 1},
    {"werkstatt_id": "W2", "standort": "Hof South", "hebebuehne": 4, "eigenbetrieb": 1},
    {"werkstatt_id": "W3", "standort": "Hof East", "hebebuehne": 3, "eigenbetrieb": 1},
    {"werkstatt_id": "W4", "standort": "Hof West", "hebebuehne": 5, "eigenbetrieb": 1},
    {"werkstatt_id": "W5", "standort": "Hof Mitte", "hebebuehne": 2, "eigenbetrieb": 0},
    {"werkstatt_id": "W6", "standort": "Hof Hafen", "hebebuehne": 2, "eigenbetrieb": 0},
]
df_wk = pd.DataFrame(werkstaette)

In [ ]:
bauteile = [
    ("Bremsbelag",        4,  320),
    ("Bremsscheibe",      6,  680),
    ("Tuersteuerung",     3,  450),
    ("Klimaanlage",       8, 1400),
    ("Batteriemodul",    12, 9800),
    ("Getriebe",         26, 14500),
    ("Motor",            40, 22000),
    ("Reifen",            2,  380),
    ("Scheibenwischer",   1,   45),
    ("Beleuchtung",       2,  120),
    ("Fahrgastsitz",      3,  260),
    ("Stromabnehmer",     5, 3100),
]

schreibweisen = {
    "Bremsbelag": ["Bremsbelag", "bremsbelag", "Bremsbelaege", "Bremsbelag VA"],
    "Tuersteuerung": ["Tuersteuerung", "Türsteuerung", "Tuer-Steuerung"],
    "Reifen": ["Reifen", "reifen", "Bereifung"],
    "Beleuchtung": ["Beleuchtung", "Beleuchtung aussen", "Licht"],
}

arten = ["Wartung", "Reparatur", "HU/AU", "Unfallschaden"]

### Werkstattaufträge generieren

In [ ]:
start = datetime(2023, 1, 1)
n = 95000
rows = []
km_aktuell = {f["fahrzeug_id"]: f["start_km_stand"] for f in fahrzeuge}

for i in range(n):
    fz = random.choice(fahrzeuge)
    bauteil, basis_std, basis_kosten = random.choices(
        bauteile, weights=[18, 8, 12, 6, 3, 2, 1, 14, 9, 11, 9, 7]
    )[0]
    art = random.choices(arten, weights=[0.42, 0.45, 0.09, 0.04])[0]

    if fz["antrieb"] != "Elektro" and bauteil in ("Batteriemodul", "Stromabnehmer"):
        bauteil = "Bremsbelag"
        basis_std, basis_kosten = 4, 320

    alter = 2026 - fz["baujahr"]
    faktor = 1 + alter * 0.025

    dauer = round(basis_std * faktor * random.uniform(0.7, 1.6), 1)
    material = round(basis_kosten * random.uniform(0.8, 1.3), 2)
    lohn = round(dauer * random.choice([78, 78, 95]), 2)

    beginn = start + timedelta(
        days=random.randint(0, 1090),
        hours=random.choice([6, 7, 8, 13, 14, 21, 22]),
    )
    ausfall_tage = max(
        1,
        int(dauer / 8)
        + random.choices([0, 1, 2, 5], weights=[0.5, 0.3, 0.15, 0.05])[0],
    )
    ende = beginn + timedelta(days=ausfall_tage)

    km_aktuell[fz["fahrzeug_id"]] += random.randint(800, 4200)
    wk = random.choices(werkstaette, weights=[0.25, 0.2, 0.15, 0.2, 0.12, 0.08])[0]

    rows.append({
        "auftrag_id": "A%07d" % i,
        "fahrzeug_id": fz["fahrzeug_id"],
        "werkstatt_id": wk["werkstatt_id"],
        "bauteil": random.choice(schreibweisen.get(bauteil, [bauteil])),
        "auftragsart": art,
        "beginn_ts": beginn.strftime("%Y-%m-%d %H:%M:%S"),
        "ende_ts": ende.strftime("%Y-%m-%d %H:%M:%S"),
        "arbeitsstunden": dauer,
        "materialkosten": material,
        "lohnkosten": lohn,
        "km_stand": km_aktuell[fz["fahrzeug_id"]],
        "status": random.choices(
            ["abgeschlossen", "Abgeschlossen", "offen", "storniert"],
            weights=[0.85, 0.07, 0.05, 0.03],
        )[0],
    })

df_au = pd.DataFrame(rows)


dupes = df_au.sample(frac=0.018, random_state=11)
df_au = pd.concat([df_au, dupes], ignore_index=True)

idx = df_au.sample(frac=0.012, random_state=12).index
df_au.loc[idx, ["beginn_ts", "ende_ts"]] = (
    df_au.loc[idx, ["ende_ts", "beginn_ts"]].to_numpy()
)

idx = df_au.sample(frac=0.008, random_state=13).index
df_au.loc[idx, "km_stand"] = [random.randint(100, 9000) for _ in idx]

idx = df_au.sample(frac=0.03, random_state=14).index
df_au.loc[idx, "materialkosten"] = None

df_au = df_au.sample(frac=1, random_state=15).reset_index(drop=True)


df_fahrz.to_csv(os.path.join(rawdata, "fahrzeuge.csv"), index=False)
df_wk.to_csv(os.path.join(rawdata, "werkstaetten.csv"), index=False)
df_au.to_csv(os.path.join(rawdata, "auftraege.csv"), index=False)

print(len(df_fahrz), len(df_wk), len(df_au))


In [ ]:
for datei in ["fahrzeuge.csv", "werkstaetten.csv", "auftraege.csv"]:
    pfad = os.path.join(rawdata, datei)
    daten = pd.read_csv(pfad)
    print(f"{datei}: {len(daten)} Zeilen")

display(pd.read_csv(os.path.join(rawdata, "auftraege.csv")).head())